# 06 · Two-stage cache (decode once, extract fast)

Reading `.log` text is the bottleneck at scale (~0.3 s/log, I/O-bound on the
external drive). This splits the work:

1. **Stage 1 - decode** each `.log` into a compact `.npz` (4 sensors + 2 wheel
   detectors, float32 ~0.7 MB vs ~12 MB text). Reads the text **once**.
2. **Stage 2 - extract features** from the `.npz` cache (~16x faster reads).
   Re-tuning feature config (`cp_decimate`, frequency range, QC) re-runs only
   stage 2 and never re-reads the text.

Put the cache on your **internal SSD** (the project folder is already on it) so
you don't copy the raw text anywhere - stage 2 then reads from SSD.

**Production note:** the serving app receives only `.log` files and calls
`extract_features(path)` directly - the cache is a training-time optimization.


In [1]:
import sys, os
from pathlib import Path
sys.path.insert(0, os.path.abspath('../src'))
import numpy as np, pandas as pd
from intelliswitch import Config, paths

# --- paths -----------------------------------------------------------------
# The project data lake (medallion tiers) is resolved by intelliswitch.paths, so
# it works regardless of the notebook's working dir and is overridable via the
# INTELLISWITCH_DATA_DIR / INTELLISWITCH_MODELS_DIR env vars.
RAW_CACHE = paths.RAW_CACHE       # data/interim/raw_cache  (decoded .npz, stage 1)
STORE     = paths.FEATURE_STORE   # data/processed/feature_store
MODELS    = paths.MODELS          # data/models
paths.ensure()                    # create the lake dirs if missing

# Raw source recordings (~7,800 logs) are large and live on the external drive;
# point DATA_ROOT at them, or symlink/copy the source into paths.RAW.
DATA_ROOT = Path(r"/Volumes/Seagate Expansion Drive/Mohammad/M Drive DTU/MATLAB Scripts/IntelliswitchScripts")
W10   = DATA_ROOT / "W10v140"       # vibration .log
BLADE = DATA_ROOT / "Bladenr8252"   # geometry .csv

cfg = Config()
cfg.cp_decimate = 1          # verify vs 1 on a sample before trusting

## Stage 1 - decode .log -> .npz (one-time; reads text once)
Resumable: re-running skips already-decoded logs. `LIMIT=None` for all.


In [2]:
from intelliswitch.io import build_raw_cache
LIMIT = 500
build_raw_cache(W10, RAW_CACHE, cfg, n_workers=os.cpu_count(), limit=LIMIT)


500 logs, 7823 already decoded, 0 to decode on 8 workers


PosixPath('/Users/Mohammad/Documents/ds-projects/intelliswitch-refactor/data/interim/raw_cache')

## Stage 2 - extract features from the cache (fast, re-runnable)
Reads the `.npz` cache. Changing feature config rebuilds this stage only.


In [3]:
from intelliswitch.pipeline import build_feature_store, load_feature_store, store_summary
build_feature_store(W10, STORE, cfg,
                    n_workers=os.cpu_count(),
                    shard_size=400,
                    limit=LIMIT,
                    raw_cache_dir=RAW_CACHE)   # <- read from stage-1 cache
store_summary(STORE)


500 logs, 400 already done, 100 to process on 8 workers
  100/100 processed | batch accepted 7/100


{'processed': 500,
 'accepted': 63,
 'feature_rows': 213,
 'top_reject_reasons': {'n_groups N != required N': 349,
  'no accepted wheel-groups': 87,
  'n_wheels(N) != N*n_groups(N)': 1}}

## Attach labels + train (instant off the cached features)


In [11]:
from intelliswitch.geometry import align_campaigns, build_deg_model
from intelliswitch.geometry.deg_model import attach_labels
from intelliswitch.model import fit_per_location

deg = build_deg_model(align_campaigns(BLADE, cfg), cfg)
data = attach_labels(load_feature_store(STORE), deg, cfg).dropna(subset=['y'])
results = fit_per_location(data, cfg)
rows = []
for name, r in results.items():
    if r['model'] is None:
        rows.append(dict(sensor=name, error=r.get('error'))); continue
    m, e = r['model'], r['eval']
    rows.append(dict(sensor=name, NOC=m.backend.n_components, n_train=m.n_train,
                     train_R2=round(m.train_r2,3), test_pts=e['n_testpoints'],
                     MAPE=round(e['mape'],1), corr=round(e['corr'],3)))
pd.DataFrame(rows)


/Users/Mohammad/Documents/ds-projects/intelliswitch-refactor/.venv/lib/python3.11/site-packages/sklearn/cross_decomposition/_pls.py:308: UserWarning: y residual is constant at iteration 0
  warnings.warn(f"y residual is constant at iteration {k}")
/Users/Mohammad/Documents/ds-projects/intelliswitch-refactor/.venv/lib/python3.11/site-packages/sklearn/cross_decomposition/_pls.py:308: UserWarning: y residual is constant at iteration 0
  warnings.warn(f"y residual is constant at iteration {k}")
/Users/Mohammad/Documents/ds-projects/intelliswitch-refactor/.venv/lib/python3.11/site-packages/sklearn/cross_decomposition/_pls.py:308: UserWarning: y residual is constant at iteration 0
  warnings.warn(f"y residual is constant at iteration {k}")
/Users/Mohammad/Documents/ds-projects/intelliswitch-refactor/.venv/lib/python3.11/site-packages/sklearn/cross_decomposition/_pls.py:308: UserWarning: y residual is constant at iteration 0
  warnings.warn(f"y residual is constant at iteration {k}")


,sensor,NOC,n_train,train_R2,test_pts,MAPE,corr
0,A1,40,7278,0.000,7,0.0,0.612
1,A2,40,2573,0.000,7,0.0,0.935
2,A7,5,124,0.776,6,1.6,0.958
3,A11,40,4051,0.464,7,3.3,0.896
